# PyDBAdminKit 0.3.0 — CLI Experiment Lab

Notebook de démonstration de la CLI `pydbadminkit` depuis Python/Jupyter.

- aucune valeur de secret n’est enregistrée ;
- les mutations réelles sont **désactivées par défaut** ;
- les opérations critiques sont montrées en dry-run uniquement ;
- ce notebook complète `00 - Setup.ipynb`, qui exerce directement l’API Python.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

from pydbadminkit.bootstrap import resolve_connection


In [ ]:
CONNECTION_PROFILE = "local-native"  # "local" pour Docker
RUN_MUTATIONS = False


def find_config_path() -> Path:
    for candidate in (Path.cwd() / "config.toml", Path.cwd().parent / "config.toml"):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("config.toml introuvable")


CONFIG_PATH = find_config_path()
PASSWORD_ENV = (
    "PYDBADMIN_NATIVE_PASSWORD"
    if CONNECTION_PROFILE == "local-native"
    else "PYDBADMIN_LOCAL_PASSWORD"
)

if not os.environ.get(PASSWORD_ENV):
    raise RuntimeError(f"Définissez {PASSWORD_ENV} avant de lancer le notebook.")

resolved_config = resolve_connection(CONNECTION_PROFILE, CONFIG_PATH)
print("Profil       :", CONNECTION_PROFILE)
print("Config       :", CONFIG_PATH)
print("Environment  :", resolved_config.environment)
print("Mutations    :", RUN_MUTATIONS)


In [ ]:
def run(args: list[str], *, expect_success: bool = True):
    base = [
        sys.executable,
        "-m",
        "pydbadminkit",
        "--connection",
        CONNECTION_PROFILE,
        "--config",
        str(CONFIG_PATH),
    ]
    command = base + args
    print("▶", " ".join(command[2:]))
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print("STDERR:", result.stderr.rstrip())
    if expect_success and result.returncode != 0:
        raise RuntimeError(f"Commande échouée avec le code {result.returncode}")
    return result


def run_json(args: list[str]):
    result = run(["--output", "json", *args])
    return json.loads(result.stdout)


def run_mutation(args: list[str]):
    if not RUN_MUTATIONS:
        print("[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)")
        return None
    if resolved_config.read_only or str(resolved_config.environment) not in {"development", "testing"}:
        raise RuntimeError("Le notebook n'autorise les mutations qu'en development/testing, read_only=false")
    return run(args)


## 1. Foundation


In [ ]:
run(["--version"])


In [ ]:
run(["connection", "test"])


In [ ]:
run(["capability", "list"])


In [ ]:
run(["server", "info"])


## 2. Object Explorer


In [ ]:
run(["database", "list"])
run(["schema", "list"])


In [ ]:
databases = run_json(["database", "list"])
print("Type:", type(databases).__name__)
print("Nombre:", len(databases))


### Objets de démonstration optionnels

La création de table/vue/index ci-dessous utilise `psycopg` uniquement si `RUN_MUTATIONS=True`.


In [ ]:
if not RUN_MUTATIONS:
    print("[SKIP] Création des objets SQL de démonstration")
else:
    import psycopg

    password = os.environ[PASSWORD_ENV]
    sql_demo = """
    CREATE TABLE IF NOT EXISTS public.customers (
        id BIGSERIAL PRIMARY KEY,
        email TEXT NOT NULL UNIQUE,
        name TEXT NOT NULL,
        created_at TIMESTAMPTZ NOT NULL DEFAULT now()
    );
    CREATE INDEX IF NOT EXISTS customers_name_idx ON public.customers(name);
    CREATE OR REPLACE VIEW public.active_customers AS
    SELECT id, email, name FROM public.customers;
    """
    with psycopg.connect(
        host=resolved_config.host,
        port=resolved_config.port,
        dbname=resolved_config.database,
        user=resolved_config.username,
        password=password,
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql_demo)
    print("Objets de démonstration créés.")


In [ ]:
run(["table", "list", "--schema", "public"])
run(["view", "list", "--schema", "public"])
run(["index", "list", "--schema", "public"])


## 3. Security — inspection


In [ ]:
run(["role", "list"])
run(["role", "list", "--login-only"])
run(["role", "describe", "postgres"])


In [ ]:
run(["access", "list", "--role", "postgres"])
run(["effective-access", "list", "--role", "postgres"])
run(["ownership", "list", "--owner", "postgres"])


## 4. Mutations — dry-run d’abord


In [ ]:
run(["--dry-run", "role", "create", "demo_user", "--login"])
critical_plan = run_json(["--dry-run", "role", "create", "demo_super", "--superuser"])
print(json.dumps(critical_plan, indent=2, ensure_ascii=False))


### Cycle réel optionnel

Activez `RUN_MUTATIONS=True` uniquement sur une base de développement. Aucun rôle `SUPERUSER` n’est créé par ce notebook.


In [ ]:
run_mutation(["--yes", "role", "create", "demo_user", "--login"])
run_mutation(["--yes", "role", "create", "demo_reader"])


In [ ]:
run_mutation(["--yes", "role", "membership-add", "demo_reader", "demo_user"])
run_mutation([
    "--yes",
    "access",
    "grant",
    "--role",
    "demo_user",
    "--object",
    "public.customers",
    "--access",
    "SELECT",
])


## 5. Sorties machine


In [ ]:
server_json = run_json(["server", "info"])
print(json.dumps(server_json, indent=2, ensure_ascii=False))
run(["--output", "yaml", "database", "list"])


## 6. Nettoyage optionnel


In [ ]:
run_mutation([
    "--yes",
    "access",
    "revoke",
    "--role",
    "demo_user",
    "--object",
    "public.customers",
    "--access",
    "SELECT",
])
run_mutation(["--yes", "role", "membership-remove", "demo_reader", "demo_user"])
run_mutation(["--yes", "role", "drop", "demo_user"])
run_mutation(["--yes", "role", "drop", "demo_reader"])


In [ ]:
if not RUN_MUTATIONS:
    print("[SKIP] Nettoyage des objets SQL de démonstration")
else:
    import psycopg

    password = os.environ[PASSWORD_ENV]
    with psycopg.connect(
        host=resolved_config.host,
        port=resolved_config.port,
        dbname=resolved_config.database,
        user=resolved_config.username,
        password=password,
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute("DROP VIEW IF EXISTS public.active_customers")
            cursor.execute("DROP TABLE IF EXISTS public.customers")
    print("Objets SQL de démonstration supprimés.")


## Commandes couvertes

| Commande | Rôle |
|---|---|
| `connection test` | tester le profil sélectionné |
| `capability list` | lister les capabilities |
| `server info` | inspecter le serveur |
| `database/schema/table/view/index` | Object Explorer |
| `role/access/effective-access/ownership` | Security read-only |
| `role create/alter/drop` | mutations de rôles |
| `role membership-add/membership-remove` | memberships |
| `access grant/revoke` | ACL relationnelles |

Les mutations réelles restent opt-in via `RUN_MUTATIONS`.
